# Magnetization collapse explorer (φ⁴)

Interactive $\Phi_m = |m|\,L^{\beta/\nu}$ vs $z = t\,L^{1/\nu}$ for the φ⁴ `L64_128` grid.

- **$\nu$, $\beta$** sliders rescale the collapse (exact 2D Ising: $\nu=1$, $\beta=1/8$).
- $\lambda_c$ defaults to 4.25; move that slider too if you want.
- Error bars are MC $\sigma(|m|)$ scaled by $L^{\beta/\nu}$.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ising.observables import (
    read_observables_for_fss,
    sigma_from_mc_batch,
)
from phi4.constants import BETA_EXACT, LAM_C, NU_EXACT
from phi4.datasets import observables_path

PLOTLY_COLORS = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
]

df = read_observables_for_fss(
    observables_path("L64_128"),
    use_m=True,
    use_m2=False,
    use_m4=False,
    use_binder=False,
    use_chi=False,
)
L_arr = df["L"].to_numpy(dtype=np.float64)
T_arr = df["T"].to_numpy(dtype=np.float64)
m_arr = np.maximum(df["magnetization"].to_numpy(dtype=np.float64), 1e-12)
sigma_m = sigma_from_mc_batch(
    df["magnetization_std"].to_numpy(dtype=np.float64),
    df["n_eff"].to_numpy(dtype=np.float64),
)
L_values = sorted(int(x) for x in np.unique(L_arr))

print(f"Loaded L64_128: {len(df)} points, L={L_values}")
print(f"exact: λ_c={LAM_C:g}, ν={NU_EXACT:g}, β={BETA_EXACT:g}")


In [ ]:
def collapse_arrays(
    *,
    lam_c: float,
    nu: float,
    beta: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (z, Phi_m, sigma_Phi) at the given exponents."""
    nu = max(float(nu), 1e-6)
    t = (T_arr - lam_c) / lam_c
    z = t * np.power(L_arr, 1.0 / nu)
    scale = np.power(L_arr, beta / nu)
    return z, m_arr * scale, sigma_m * scale


def _build_figure(lam_c: float, nu: float, beta: float) -> go.FigureWidget:
    z, phi, sigma_phi = collapse_arrays(lam_c=lam_c, nu=nu, beta=beta)
    traces = []
    for i, Lval in enumerate(L_values):
        mask = L_arr == float(Lval)
        order = np.argsort(z[mask])
        traces.append(
            go.Scatter(
                x=z[mask][order].tolist(),
                y=phi[mask][order].tolist(),
                error_y=dict(
                    type="data",
                    array=sigma_phi[mask][order].tolist(),
                    visible=True,
                    thickness=1.2,
                    width=3,
                ),
                mode="markers",
                marker=dict(
                    size=9,
                    color=PLOTLY_COLORS[i % len(PLOTLY_COLORS)],
                    opacity=0.9,
                ),
                name=f"L={Lval}",
                customdata=np.column_stack(
                    [T_arr[mask][order], m_arr[mask][order]]
                ).tolist(),
                hovertemplate=(
                    f"L={Lval}<br>λ=%{{customdata[0]:.4f}}"
                    "<br>z=%{x:.3g}<br>Φ_m=%{y:.4f}"
                    "<br>|m|=%{customdata[1]:.4f}<extra></extra>"
                ),
            )
        )
    y_lo = float(np.min(phi - sigma_phi)) - 0.05
    y_hi = float(np.max(phi + sigma_phi)) + 0.05
    traces.append(
        go.Scatter(
            x=[0.0, 0.0],
            y=[y_lo, y_hi],
            mode="lines",
            line=dict(color="gray", width=1, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        )
    )
    return go.FigureWidget(
        data=traces,
        layout=go.Layout(
            title=dict(
                text=(
                    f"φ⁴ L64_128 · Φ_m collapse · "
                    f"λ_c={lam_c:.4f}, ν={nu:.3g}, β={beta:.4g}"
                ),
                font=dict(size=13),
            ),
            xaxis_title="z = t L^(1/ν)",
            yaxis_title="Φ_m = |m| L^(β/ν)",
            yaxis=dict(range=[y_lo, y_hi]),
            width=780,
            height=480,
            template="plotly_white",
            margin=dict(l=56, r=16, t=48, b=48),
            legend=dict(
                font=dict(size=10),
                yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99,
            ),
            hovermode="closest",
        ),
    )


fig = _build_figure(LAM_C, NU_EXACT, BETA_EXACT)


def _update_collapse(lam_c: float, nu: float, beta: float) -> None:
    z, phi, sigma_phi = collapse_arrays(lam_c=lam_c, nu=nu, beta=beta)
    y_lo = float(np.min(phi - sigma_phi)) - 0.05
    y_hi = float(np.max(phi + sigma_phi)) + 0.05
    with fig.batch_update():
        for i, Lval in enumerate(L_values):
            mask = L_arr == float(Lval)
            order = np.argsort(z[mask])
            tr = fig.data[i]
            tr.x = z[mask][order].tolist()
            tr.y = phi[mask][order].tolist()
            tr.error_y.array = sigma_phi[mask][order].tolist()
            tr.customdata = np.column_stack(
                [T_arr[mask][order], m_arr[mask][order]]
            ).tolist()
        # vertical z=0 guide is the last trace
        fig.data[-1].y = [y_lo, y_hi]
        fig.layout.yaxis.range = [y_lo, y_hi]
        fig.layout.title.text = (
            f"φ⁴ L64_128 · Φ_m collapse · "
            f"λ_c={lam_c:.4f}, ν={nu:.3g}, β={beta:.4g}"
        )


In [ ]:
slider_lam = widgets.FloatSlider(
    value=LAM_C,
    min=4.15,
    max=4.35,
    step=0.001,
    description="λ_c",
    readout_format=".4f",
    continuous_update=True,
    layout=widgets.Layout(width="420px"),
)
slider_nu = widgets.FloatSlider(
    value=NU_EXACT,
    min=0.5,
    max=1.5,
    step=0.005,
    description="ν",
    readout_format=".3f",
    continuous_update=True,
    layout=widgets.Layout(width="420px"),
)
slider_beta = widgets.FloatSlider(
    value=BETA_EXACT,
    min=0.05,
    max=0.25,
    step=0.001,
    description="β",
    readout_format=".4f",
    continuous_update=True,
    layout=widgets.Layout(width="420px"),
)
btn_exact = widgets.Button(description="Reset to exact", button_style="info")
btn_laps = widgets.Button(
    description="Set to LAPS mean",
    tooltip="λ_c≈4.247, ν≈0.752 from m-only LAPS fit",
)
status = widgets.HTML(
    value=(
        f"<b>exact</b>: λ_c={LAM_C:g}, ν={NU_EXACT:g}, β={BETA_EXACT:g} "
        f"&nbsp;|&nbsp; drag sliders to watch collapse"
    )
)


def _on_change(_change=None) -> None:
    _update_collapse(slider_lam.value, slider_nu.value, slider_beta.value)


def _reset_exact(_btn=None) -> None:
    slider_lam.value = LAM_C
    slider_nu.value = NU_EXACT
    slider_beta.value = BETA_EXACT


def _set_laps(_btn=None) -> None:
    # From L64_128 m-only LAPS (no corr/disc), β fixed.
    slider_lam.value = 4.2474
    slider_nu.value = 0.752
    slider_beta.value = BETA_EXACT


for s in (slider_lam, slider_nu, slider_beta):
    s.observe(_on_change, names="value")
btn_exact.on_click(_reset_exact)
btn_laps.on_click(_set_laps)

ui = widgets.VBox(
    [
        status,
        widgets.HBox([slider_nu, slider_beta]),
        widgets.HBox([slider_lam, btn_exact, btn_laps]),
        fig,
    ]
)
display(ui)
